# MinerU 2.5 Pro (1.2B) Benchmark: Layout-Aware vs. Direct Full-Page
**Pierce 1890 Medical Adviser · Team G07 · A2 SOTA Vision-Language Document Parser**

Evaluates **opendatalab/MinerU2.5-Pro-2604-1.2B** using the official **`MinerUClient`** pipeline across 24 test pages in two paradigms:
1. **Mode 1 (Layout-Aware)**: PP-DocLayoutV3 bounding box text crops (`is_figure=False`).
2. **Mode 2 (Direct Full-Page)**: Un-cropped 300 DPI high-resolution page images.
3. **Comparative Evaluation**: Side-by-side CER, WER, and Word F1 score comparison with CSV & Markdown reports.


## Cell 1 — Download Micromamba & Create Isolated Environment (`/kaggle/working/mamba_env`)
Installs Python 3.10, PyTorch 2.5.1 (CUDA 12.4), `mineru-vl-utils[transformers]==1.0.5`, `transformers==4.57.6`, `accelerate`, etc.

In [ ]:
import os, subprocess, sys
from pathlib import Path

MAMBA_BIN = Path("/tmp/bin/micromamba")
ENV_DIR   = Path("/kaggle/working/mamba_env")
PY_BIN    = ENV_DIR / "bin" / "python"
PIP_BIN   = ENV_DIR / "bin" / "pip"

# Step 1: Download micromamba binary
if not MAMBA_BIN.exists():
    print("Downloading Micromamba binary...")
    Path("/tmp/bin").mkdir(parents=True, exist_ok=True)
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba",
        shell=True, check=True
    )
    print(f"Micromamba installed to {MAMBA_BIN}")

# Step 2: Create isolated Micromamba environment
if not PY_BIN.exists():
    print(f"Creating Micromamba environment at {ENV_DIR}...")
    subprocess.run([
        str(MAMBA_BIN), "create", "-y", "-p", str(ENV_DIR),
        "-c", "conda-forge", "python=3.10", "pip"
    ], check=True)
    print("Installing PyTorch 2.5.1 (CUDA 12.4)... ")
    subprocess.run([
        str(PIP_BIN), "install", "-q",
        "torch==2.5.1", "torchvision==0.20.1",
        "--extra-index-url", "https://download.pytorch.org/whl/cu124"
    ], check=True)
    print("Installing MinerU-VL-Utils & Official Pipeline dependencies...")
    subprocess.run([
        str(PIP_BIN), "install", "-q",
        "mineru-vl-utils[transformers]==1.0.5",
        "transformers==4.57.6",
        "accelerate",
        "sentencepiece",
        "protobuf",
        "pillow",
        "pymupdf",
        "jiwer",
        "opencv-python-headless"
    ], check=True)
    print("All packages installed successfully in MinerU Micromamba environment!")
else:
    print(f"Micromamba environment already exists at {ENV_DIR}")

# Verify environment
subprocess.run([
    str(PY_BIN), "-c",
    "import torch, transformers, mineru_vl_utils; print('Torch CUDA:', torch.cuda.is_available(), 'Torch Version:', torch.__version__); print('Transformers:', transformers.__version__); print('MinerU-VL-Utils: OK')"
])


## Cell 2 — Verify Dataset Paths

In [ ]:
DET = Path("/kaggle/input/datasets/kmazd1110/ocr-layout-dataset/ocr-layout-dataset/ppdoclayout-v3/detections.jsonl")
GT = Path("/kaggle/input/datasets/kmazd1110/dl-ocr-test-dataset/ocr-gt-labels/labels.jsonl")
IMAGES = Path("/kaggle/input/datasets/kmazd1110/dl-ocr-test-dataset/ocr-gt-labels/heldout_pages")
PDF = Path("/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor/EN_The-Peoples-Common-Sense-Medical-Adviser.pdf")

print("=" * 80)
print("VERIFYING DATASET PATHS:")
for p, name in [(DET, "PP-DocLayoutV3 Detections"), (GT, "Ground Truth Labels"), (IMAGES, "Heldout Page Images"), (PDF, "PDF Document")]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status:7s}] {name:25s} -> {p}")
print("=" * 80)


## Cell 3 — Launch MinerU 2.5 Pro Benchmark via Micromamba (`mamba_env/bin/python`)

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path

MAMBA_PY = Path("/kaggle/working/mamba_env/bin/python")
SCRIPT_PATH = Path("/kaggle/working/mineru_bench.py")

SCRIPT_B64 = "..."
SCRIPT_PATH.write_bytes(base64.b64decode(SCRIPT_B64))
print(f"Benchmark script written to {SCRIPT_PATH} ({SCRIPT_PATH.stat().st_size} bytes)")
print("Starting MinerU 2.5 Pro benchmark (running Mode 1 & Mode 2 across 24 test pages on GPU)...")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["CUDA_VISIBLE_DEVICES"] = "0"

# Run unbuffered with real-time output streaming
process = subprocess.Popen(
    [str(MAMBA_PY), "-u", str(SCRIPT_PATH)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
print("Return code:", process.returncode)


## Cell 4 — Display Summary Benchmark Report

In [ ]:
report_file = Path("/kaggle/working/mineru_results/report.md")
if report_file.exists():
    print(report_file.read_text(encoding="utf-8"))
else:
    print("Report not found at:", report_file)
